## Popularity recommendation
В этом методы мы будем предлагать пользователям самые высокооцененные фильмы и сериалы. То есть, мы отказываемся от персональных рекоммендаций, каждый пользователь будет получать в рекоммендациях одни и те же предложения.

Этот метод мы будем использовать в качестве baseline решения, далее все результаты мы будем сравнивать с результатами этого метода.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data_loader import DataLoader
import pandas as pd

In [2]:
loader = DataLoader(path='../data')
ratings, movies, users = loader.load_all()

In [3]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,2000-12-31 22:12:40
1,1,661,3,2000-12-31 22:35:09
2,1,914,3,2000-12-31 22:32:48
3,1,3408,4,2000-12-31 22:04:35
4,1,2355,5,2001-01-06 23:38:11


In [4]:
movie_stats = (
    ratings
    .groupby("movieId")
    .agg({
        "rating": ["mean", "count"]
    })
)

movie_stats.columns = ["mean_rating", "rating_count"]
movie_stats = movie_stats.reset_index()

In [5]:
movie_stats.head()

,movieId,mean_rating,rating_count
0,1,4.146846,2077
1,2,3.201141,701
2,3,3.016736,478
3,4,2.729412,170
4,5,3.006757,296


In [6]:
min_count = 50
popular_movies = movie_stats[movie_stats["rating_count"] >= min_count]
popular_movies.head()

,movieId,mean_rating,rating_count
0,1,4.146846,2077
1,2,3.201141,701
2,3,3.016736,478
3,4,2.729412,170
4,5,3.006757,296


In [7]:
popular_movies = popular_movies.sort_values(by="mean_rating", ascending=False)
popular_movies.head()

,movieId,mean_rating,rating_count
2698,2905,4.608696,69
1839,2019,4.560510,628
309,318,4.554558,2227
802,858,4.524966,2223
708,745,4.520548,657


In [8]:
popular_movies = popular_movies.merge(movies.drop(columns="genres", axis=1), on="movieId")
popular_movies.head()

,movieId,mean_rating,rating_count,title
0,2905,4.608696,69,Sanjuro (1962)
1,2019,4.560510,628,Seven Samurai (The Magnificent Seven) (Shichin...
2,318,4.554558,2227,"Shawshank Redemption, The (1994)"
3,858,4.524966,2223,"Godfather, The (1972)"
4,745,4.520548,657,"Close Shave, A (1995)"


In [9]:
def get_recommendations(k=10): 
    return popular_movies["movieId"].head(k).tolist()

In [12]:
from src.metrics import Metrics

metrics = Metrics()

recommended = get_recommendations(10)

total_prec = 0
total_rec = 0
total_ndcg = 0

for user_id in users["userId"].unique():

    relevant = ratings[
        (ratings.userId == user_id) &
        (ratings.rating >= 4)
    ]["movieId"].tolist()

    precision, recall, ndcg = metrics.get_all(
        recommended,
        relevant,
        10
    )

    total_prec += precision
    total_rec += recall
    total_ndcg += ndcg

uniq_users = users['userId'].unique().shape[0]

print(f"total precision: {total_prec / uniq_users}\ntotal recall: {total_rec / uniq_users}\ntotal ndcg: {total_ndcg / uniq_users}")

total precision: 0.20639072847682174
total recall: 0.028203429510127245
total ndcg: 0.1766373435196049


## Выводы
Метрики Precision и ndcg показали минимально приемлемый результат, в случае Recall ситуация противоположная.

Результат Precision, по моему мнению, связаны с тем, что очень высокооцененные фильмы с большей вероятностью будут в топе любимых у пользователя, ведь их популярность должна быть чем-то обусловлена. 

Низкие показатели Recall появляются из-за того, что в моей постановке задачи и способе подсчета метрик Recall всегда будет иметь относительно низкие значения. Мой способ подсчета метрик для k рекомендаций всегда усредняется на размер релевантной выборки, отчего более менее приличные значения метрик можно получить только в случае, когда размер релевантной выборки будет равен размеру выборки рекомендаций. А учитывая мой способ выбора релевантных объектов, добиться желаемого результата не представляется возможным.

Метрику ndcg сложнее интерпретировать, ведь она зависит еще и от правильности ранжирования объектов рекомендаций. Полученные результаты, как минимум, указывают на то, что в рекомендованной выборке содержались релевантные объекты. Здесь аналогично пункту про Precision